# 🎼 Notebook 2: Choreography vs Orchestration

Two ways to coordinate the steps of a saga:

### Orchestration
A single **orchestrator** calls each service in sequence and decides what to do on failure.
- ✅ Easy to see the full flow.
- ❌ Orchestrator is a central dependency.

### Choreography
Services **react to events** published by peers. Nobody is in charge — they're all listening.
- ✅ Decoupled, scales well.
- ❌ Flow is implicit; harder to debug.

## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Orchestrator (from notebook 1) — already shown.

## Choreography with a tiny event bus

In [ ]:
from collections import defaultdict

class Bus:
    def __init__(self):
        self.subs = defaultdict(list)
    def on(self, event, fn):
        self.subs[event].append(fn)
    def emit(self, event, **data):
        print(f'  📨 {event} {data}')
        for fn in self.subs[event]:
            fn(**data)

bus = Bus()
state = {'stock':10, 'charged':0, 'shipment': None}

def on_order_placed(order_id, **_):
    state['stock'] -= 1
    bus.emit('stock_reserved', order_id=order_id)

def on_stock_reserved(order_id, **_):
    state['charged'] = 20
    bus.emit('payment_ok', order_id=order_id)

def on_payment_ok(order_id, **_):
    state['shipment'] = 'booked'
    bus.emit('shipped', order_id=order_id)

bus.on('order_placed',   on_order_placed)
bus.on('stock_reserved', on_stock_reserved)
bus.on('payment_ok',     on_payment_ok)

bus.emit('order_placed', order_id=1)
print('final state:', state)


**Key difference**: in choreography, there is no central saga.py. The workflow is emergent — it's the set of subscriptions.